# UFUU Higgs Parameter Fixing — Verification Notebook

**Repository file**: `UFUU_Higgs_Verification.ipynb`  
**Companion script**: `UFUU_Higgs_Verification.py`  
**Cited in**: Tuttle, W. J. (2026), Section 7.1.1

This notebook implements the exact grid search procedure described in Section 7.1.1 of the
foundation paper. It confirms that under the linear eigenvalue constraint λ_sym = 0.259,
the Landau potential coefficient μ² = 1 − λ_sym = +0.741 throughout the constrained
parameter slice, establishing that the double-well condition (μ² < 0) is not achieved
within this parameterization. This result is reported honestly in the paper as an
open quantitative question whose resolution constitutes a genuine test of the framework.

**Expected output**: μ² ≈ +0.741 (single-well). The notebook does NOT reproduce a
Mexican-hat potential — it confirms that reproducing it requires either relaxing the
symmetric-point constraint or deriving the attractor m from first principles.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Imports complete.')

## 1. Constants and Fixed-Point Values

In [ ]:
# LAMBDA_SYM: dominant eigenvalue of the linearized UFUU transfer operator
# under information maximization. Derived analytically in Tuttle (2026a), Section 2.
# Value: lambda_sym = 2^(-1/12) in log-scaling metric, approx 0.259 in the
# coordinate system used for the order-parameter map.
LAMBDA_SYM = 0.259

# ATTRACTOR_M: the background mean value at which the linear eigenvalue constraint
# is evaluated. Set to 0.0 for evaluation at the symmetric fixed point.
# Note: the value 1.134663 appearing in earlier drafts was derived from
# unverified external calculations and is NOT used here. The symmetric-point
# evaluation (m=0) is the correct baseline for the linearized analysis.
ATTRACTOR_M = 0.0   # symmetric fixed point

# Note on m=0 vs m>0:
# At m=0: linear constraint gives 2*alpha = lambda_sym => alpha = lambda_sym/2 = 0.1295
# At m>0: alpha and beta jointly satisfy 2*alpha + 2*beta*m^2 = lambda_sym
# The double-well condition requires mu^2 = 1 - lambda_sym < 0, i.e. lambda_sym > 1.
# Since lambda_sym = 0.259 < 1, mu^2 = +0.741 throughout this slice.
# This is the key finding reported in the paper.

N_SAMPLES = 10000
N_ITER = 12
PHI_STD = 0.01

print(f'LAMBDA_SYM  = {LAMBDA_SYM}')
print(f'ATTRACTOR_M = {ATTRACTOR_M}  (symmetric fixed point)')
print(f'mu^2 = 1 - lambda_sym = {1 - LAMBDA_SYM:.4f}  (positive => single-well)')

## 2. Effective Recursion Map (from Taylor Expansion)

In [ ]:
# From Taylor expansion of F(m+phi, m-phi) around phi=0 (verified in SymPy):
# phi_{n+1} = (2*alpha + 2*beta*m^2)*phi_n + (-2*beta + 8*gamma)*phi_n^3 + O(phi^5)
#
# Gradient flow identification gives:
#   mu^2   = 1 - (2*alpha + 2*beta*m^2)
#   lambda = 2*beta - 8*gamma
# (See UFUU_Higgs_Verification.py for SymPy symbolic proof)

def effective_map(phi, alpha, beta, gamma, m=0.0):
    """Effective recursion map for the order parameter phi."""
    linear_coeff = 2 * alpha + 2 * beta * m**2
    cubic_coeff  = -2 * beta + 8 * gamma
    return linear_coeff * phi + cubic_coeff * phi**3

print('effective_map defined.')

## 3. Parameter Grid Under Linear Eigenvalue Constraint

In [ ]:
# At m=0: linear constraint 2*alpha = lambda_sym => alpha = lambda_sym/2
# alpha is therefore fixed; grid search is over (beta, gamma) only.
alpha_fixed = LAMBDA_SYM / 2
print(f'alpha fixed by eigenvalue constraint: alpha = lambda_sym/2 = {alpha_fixed:.6f}')

beta_range  = np.linspace(-0.05, 0.05, 21)
gamma_range = np.linspace(0.0, 0.2, 21)

# Stability condition: lambda = 2*beta - 8*gamma > 0
def lambda_coeff(b, g):
    return 2 * b - 8 * g

stable_mask = lambda_coeff(beta_range[:, None], gamma_range) > 0
n_stable = stable_mask.sum()
print(f'Parameter grid: {len(beta_range)} x {len(gamma_range)} = {len(beta_range)*len(gamma_range)} points')
print(f'Stable points (lambda > 0): {n_stable}')

## 4. Initial Distribution and Entropy Function

In [ ]:
np.random.seed(42)
phi0 = np.random.normal(0, PHI_STD, N_SAMPLES)
print(f'Initial distribution: N={N_SAMPLES}, std={PHI_STD}')

def compute_final_entropy(alpha, beta, gamma, m=0.0, n_iter=N_ITER):
    """Iterate the order-parameter map and compute histogram-based Shannon entropy."""
    phi = phi0.copy()
    for _ in range(n_iter):
        phi = effective_map(phi, alpha, beta, gamma, m)
        phi = phi[np.isfinite(phi)] if np.any(~np.isfinite(phi)) else phi
    if len(phi) < 10:
        return -np.inf
    hist, bin_edges = np.histogram(phi, bins=100, density=True)
    hist = hist[hist > 1e-12]
    bin_width = bin_edges[1] - bin_edges[0]
    return -np.sum(hist * np.log2(hist)) * bin_width

print('compute_final_entropy ready.')

## 5. Grid Search — Entropy Maximization

In [ ]:
entropies = np.full((len(beta_range), len(gamma_range)), -np.inf)
max_entropy = -np.inf
best_alpha = best_beta = best_gamma = None

for i, beta in enumerate(beta_range):
    for j, gamma in enumerate(gamma_range):
        if not stable_mask[i, j]:
            continue
        ent = compute_final_entropy(alpha_fixed, beta, gamma)
        entropies[i, j] = ent
        if ent > max_entropy:
            max_entropy = ent
            best_alpha, best_beta, best_gamma = alpha_fixed, beta, gamma

print('Grid search complete.')
print(f'Best: alpha={best_alpha:.5f}, beta={best_beta:.5f}, gamma={best_gamma:.5f}')
print(f'Max entropy = {max_entropy:.4f}')

## 6. Higgs Potential Coefficients

In [ ]:
mu2          = 1 - (2 * best_alpha + 2 * best_beta * ATTRACTOR_M**2)
lambda_higgs = 2 * best_beta - 8 * best_gamma

print('=== UFUU Higgs Parameter Fixing Results ===')
print(f'  alpha = {best_alpha:.5f}')
print(f'  beta  = {best_beta:.5f}')
print(f'  gamma = {best_gamma:.5f}')
print(f'  Max Shannon entropy = {max_entropy:.4f}')
print()
print(f'  mu^2   = {mu2:.5f}  (positive => symmetric point is STABLE)')
print(f'  lambda = {lambda_higgs:.5f}  (positive => broken vacua are stable)')
print()
print(f'Double-well condition (mu^2 < 0): NOT SATISFIED')
print(f'Reason: mu^2 = 1 - lambda_sym = 1 - {LAMBDA_SYM} = {1-LAMBDA_SYM:.3f} identically')
print(f'        under the linear eigenvalue constraint.')

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: entropy heatmap
valid = entropies.copy()
valid[valid == -np.inf] = np.nan
im = axes[0].imshow(valid, origin='lower', aspect='auto',
                    extent=[gamma_range[0], gamma_range[-1],
                            beta_range[0],  beta_range[-1]],
                    cmap='viridis')
plt.colorbar(im, ax=axes[0], label='Shannon entropy')
axes[0].set_xlabel('gamma')
axes[0].set_ylabel('beta')
axes[0].set_title('Entropy over (beta, gamma) grid\n(alpha fixed by eigenvalue constraint)')
if best_beta is not None:
    axes[0].scatter([best_gamma], [best_beta], c='red', s=100, zorder=5, label='Maximum')
    axes[0].legend()

# Right: Landau potential at best parameters
phi_range = np.linspace(-2, 2, 500)
V = 0.5 * mu2 * phi_range**2 + 0.25 * lambda_higgs * phi_range**4
axes[1].plot(phi_range, V, 'b-', linewidth=2)
axes[1].axhline(0, color='k', linewidth=0.5)
axes[1].axvline(0, color='k', linewidth=0.5)
axes[1].set_xlabel('phi (order parameter)')
axes[1].set_ylabel('V(phi)')
axes[1].set_title(f'Landau potential at best parameters\n'
                  f'mu^2 = {mu2:.3f} (single-well: mu^2 > 0)')
axes[1].set_ylim(-1, 5)

plt.tight_layout()
plt.savefig('higgs_entropy_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to higgs_entropy_grid.png')

## 8. Save Results

In [ ]:
np.savez('higgs_entropy_grid.npz',
         beta=beta_range, gamma=gamma_range, entropies=entropies,
         best_alpha=best_alpha, best_beta=best_beta, best_gamma=best_gamma,
         mu2=mu2, lambda_higgs=lambda_higgs)
print('Results saved to higgs_entropy_grid.npz')

## Conclusion

The grid search under the linear eigenvalue constraint λ_sym = 0.259 yields
**μ² = 1 − λ_sym = +0.741** throughout the constrained parameter slice.

This is not a failure of the search — it is a structural result. The constraint
pins 2α + 2βm² = 0.259, so μ² = 1 − 0.259 = 0.741 identically, regardless of
how β and γ are chosen within the stable region.

**The double-well condition μ² < 0 requires λ_sym > 1.** Since the linearized
transfer operator gives λ_sym = 0.259, achieving the Mexican-hat configuration
requires one of:

1. Relaxing the symmetric-point constraint (m > 0, derived from first principles)
2. A different or refined entropy functional that selects a different eigenvalue regime

As stated in Tuttle (2026), Section 7.1.1:
> *"The decisive implication is that the sign of μ² is not a tunable parameter
> but a prediction of the fold itself: determining whether the information-maximization
> principle drives the system into the μ² < 0 regime becomes a single, sharp test
> that decides whether the UFUU recursion truly reproduces electroweak symmetry
> breaking or fails to do so."*

This notebook is fully reproducible. Seed: 42. All results are deterministic.
